In [ ]:
Install tensorflow, matplotlib and mount the google drive

In [ ]:
!pip install tensorflow matplotlib -q
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

In [ ]:
Load the raw data and build richer aggregation 

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("/content/drive/MyDrive/chicago-crime-ml/data/model1_classification.csv")
print(f"Loaded {len(df_raw):,} rows")
print(df_raw.columns.tolist())

agg = df_raw.groupby(["district", "year", "month"]).agg(
    crime_count        = ("year", "count"),
    arrest_rate        = ("arrest_int", "mean"),
    domestic_rate      = ("domestic_int", "mean"),
    avg_hour           = ("hour_of_day", "mean"),
    rush_hour_rate     = ("is_rush_hour", "mean"),
    weekend_rate       = ("is_weekend", "mean"),
    unique_wards       = ("ward", "nunique"),
    unique_communities = ("community_area", "nunique"),
    top_location       = ("location_encoded", lambda x: x.mode()[0]),
).reset_index()

agg["season"] = agg["month"].map({
    12: 1, 1: 1, 2: 1,
     3: 2, 4: 2, 5: 2,
     6: 3, 7: 3, 8: 3,
     9: 4, 10: 4, 11: 4
})

agg = agg.sort_values(["district", "year", "month"]).reset_index(drop=True)

agg["crime_count_lag1"] = agg.groupby("district")["crime_count"].shift(1)
agg["crime_count_lag2"] = agg.groupby("district")["crime_count"].shift(2)
agg["crime_count_lag3"] = agg.groupby("district")["crime_count"].shift(3)

agg["rolling_avg_3m"] = (
    agg.groupby("district")["crime_count"]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

agg["crime_count_yoy"] = agg.groupby(["district", "month"])["crime_count"].shift(1)

agg = agg.dropna()

print(f"\nAggregated dataset: {len(agg):,} rows")
print(f"Crime count range: {agg['crime_count'].min()} – {agg['crime_count'].max()}")
print(f"Mean crime count: {agg['crime_count'].mean():.1f}")
print(f"\nFeatures: {agg.columns.tolist()}")
print(agg.head())

In [ ]:
Prepare the features 

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = [
    "district", "year", "month", "season",
    "arrest_rate", "domestic_rate",
    "avg_hour", "rush_hour_rate", "weekend_rate",
    "unique_wards", "unique_communities", "top_location",
    "crime_count_lag1", "crime_count_lag2", "crime_count_lag3",
    "rolling_avg_3m",
    "crime_count_yoy",
]

X = agg[feature_cols].values.astype("float32")
y = agg["crime_count"].values.astype("float32")

train_mask = agg["year"] <= 2021
X_train, y_train = X[train_mask],  y[train_mask]
X_test,  y_test  = X[~train_mask], y[~train_mask]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train years: 2015–2021 | Test years: 2022–2023")
print(f"Features: {len(feature_cols)}")

In [ ]:
Build the model 

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.1),
    keras.layers.Dense(1)
])

model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="huber",
    metrics=["mae"]
)

In [ ]:
Train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=6, verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
Print out results

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_pred = model.predict(X_test).flatten()
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2)

print(f"\nTest MAE:  {mae:.2f} crimes")
print(f"Test RMSE: {rmse:.2f} crimes")
print(f"R² score:  {r2:.4f}")
print(f"Mean crime count in test set: {y_test.mean():.2f}")
print(f"MAE as % of mean: {mae / y_test.mean() * 100:.1f}%")

results = pd.DataFrame({
    "district":  agg[~train_mask]["district"].values[:15],
    "year":      agg[~train_mask]["year"].values[:15],
    "month":     agg[~train_mask]["month"].values[:15],
    "actual":    y_test[:15].astype(int),
    "predicted": np.round(y_pred[:15]).astype(int)
})
print("\nSample predictions vs actuals:")
print(results.to_string(index=False))

In [ ]:
XGBoost comparison 

In [ ]:
!pip install xgboost -q
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = xgb.predict(X_test)
mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb   = 1 - np.sum((y_test - y_pred_xgb)**2) / np.sum((y_test - np.mean(y_test))**2)

print("=" * 40)
print("Model Comparison")
print("=" * 40)
print(f"{'Metric':<12} {'TensorFlow':>12} {'XGBoost':>12}")
print("-" * 40)
print(f"{'MAE':<12} {mae:>12.2f} {mae_xgb:>12.2f}")
print(f"{'RMSE':<12} {rmse:>12.2f} {rmse_xgb:>12.2f}")
print(f"{'R²':<12} {r2:>12.4f} {r2_xgb:>12.4f}")
print(f"{'MAE % mean':<12} {mae/y_test.mean()*100:>11.1f}% {mae_xgb/y_test.mean()*100:>11.1f}%")

In [ ]:
Training curves 

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["loss"],     label="Train Loss")
ax1.plot(history.history["val_loss"], label="Val Loss")
ax1.set(title="Training Loss (Huber)", xlabel="Epoch", ylabel="Loss")
ax1.legend()
ax2.plot(history.history["mae"],     label="Train MAE")
ax2.plot(history.history["val_mae"], label="Val MAE")
ax2.set(title="Mean Absolute Error", xlabel="Epoch", ylabel="MAE")
ax2.legend()
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/chicago-crime-ml/outputs/tf_training_curves.png", dpi=150)
plt.show()

In [ ]:
Actual vs predicted 

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, s=15)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         "r--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual crime count")
plt.ylabel("Predicted crime count")
plt.title("Actual vs Predicted Crime Count (TF Regressor v3)")
plt.legend()
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/chicago-crime-ml/outputs/tf_actual_vs_predicted.png", dpi=150)
plt.show()

In [ ]:
Save the model 

In [ ]:
model.save("/content/drive/MyDrive/chicago-crime-ml/outputs/tf_crime_regressor.keras")
print("Saved model to Google Drive")